# PhishGuard-PK | XLM-RoBERTa Multilingual Training
**Step A:** Google Colab > Runtime > Change runtime type > select GPU. **Step B:** Upload the v2 project ZIP. **Step C:** Run all cells.

This trains an advanced model using development labels which include synthetic and provisional sources, so any metrics are **development-only**. A university-valid result still requires independent verified labels. The saved checkpoint is needed for the XLM-R model mode.

In [ ]:
from google.colab import files
from pathlib import Path
import os,sys,zipfile
up=files.upload();archives=[n for n in up if n.lower().endswith(".zip")];assert archives,"Upload PHISHGUARD_V2_FULL_SOFTWARE.zip"
root=Path("/content/PhishGuard_PK_v2_Full_Software");root.mkdir(parents=True,exist_ok=True)
with zipfile.ZipFile(archives[0]) as z:
 if any(".." in Path(n).parts or n.startswith("/") for n in z.namelist()):raise ValueError("Unsafe archive")
 z.extractall(root)
os.chdir(root);sys.path.insert(0,str(root))

In [ ]:
%pip -q install -r /content/PhishGuard_PK_v2_Full_Software/requirements-advanced.txt

In [ ]:
import torch
print("CUDA available:",torch.cuda.is_available())
if not torch.cuda.is_available():
 raise RuntimeError("GPU required for this notebook. Select GPU under Runtime > Change runtime type.")
print("GPU:",torch.cuda.get_device_name(0))

In [ ]:
import subprocess,sys
subprocess.run([sys.executable,"scripts/train_transformer.py"],check=True)
print("XLM-R training and provisional evaluation finished")

In [ ]:
from google.colab import files
import zipfile
with zipfile.ZipFile("PHISHGUARD_XLMR_MODEL_OUTPUT.zip","w",zipfile.ZIP_DEFLATED) as z:
 for path in (root/"models/xlmr_development").rglob("*"):
  if path.is_file():z.write(path,path.relative_to(root))
 if (root/"reports/xlmr_optional_metrics.json").exists():z.write(root/"reports/xlmr_optional_metrics.json","reports/xlmr_optional_metrics.json")
files.download("PHISHGUARD_XLMR_MODEL_OUTPUT.zip")